In [3]:
import torch.nn as nn
import torch
import pandas as pd
# añadidos
from sklearn.preprocessing import StandardScaler
import numpy as np
import torch.utils.data as torch_data

In [4]:
data = pd.read_csv("Pokemon.csv")
data.head()

,#,Name,Type 1,Type 2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
0,1,Bulbasaur,Grass,Poison,318,45,49,49,65,65,45,1,False
1,2,Ivysaur,Grass,Poison,405,60,62,63,80,80,60,1,False
2,3,Venusaur,Grass,Poison,525,80,82,83,100,100,80,1,False
3,3,VenusaurMega Venusaur,Grass,Poison,625,80,100,123,122,120,80,1,False
4,4,Charmander,Fire,NaN,309,39,52,43,60,50,65,1,False


## Descripción de la tarea

Trabajaremos con un dataset de la serie "Pokémon" que desglosa las capacidades de cada criatura en atributos cuantitativos. La tarea consiste en utilizar arquitecturas de redes neuronales simples (MLP) para identificar patrones que distingan a los Pokémon Legendarios del resto. Deberán procesar estas variables y entrenar un clasificador que maximice la capacidad predictiva sobre la variable objetivo "Legendary".

Consideraciones:
- Debe entregarlo a más tardar el 29 de mayo a las 18:00 horas.
- Debe ser entregado al correo luis.llanca@uach.cl con el asunto "Tarea-1-MLP", el archivo debe llamarse NG-MLP-Tarea1.ipynb donde NG es el número de grupo. Es importante que el asunto sea exactamente el mismo. También, se les pedirá que se anoten en la plantilla (que se compartirá posteriormente) para una pequeña interrogación.
- Por cada 20 minutos de retraso, se descontará una décima de la nota. 
- Si necesitan ayuda, pueden escribir a los correos luis.llanca@uach.cl, luis.llanca@cenia.cl o escribir al discord de usuario: llanking (tengo una foto de mi gata). PD: prefiero mucho más el discord. 

### Explicación del dataset
Explique el dataset en detalle, incluyendo como mínimo una pequeña descripción de cada columna, el tipo de datos que contiene cada columna, y cualquier información adicional relevante para entender el dataset.

In [ ]:
# Se usaron los siguientes métodos para analizar el dataset jeje
# data.info()
# data.describe()
# data.isnull().sum()

,#,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation
count,800.000000,800.00000,800.000000,800.000000,800.000000,800.000000,800.000000,800.000000,800.00000
mean,362.813750,435.10250,69.258750,79.001250,73.842500,72.820000,71.902500,68.277500,3.32375
std,208.343798,119.96304,25.534669,32.457366,31.183501,32.722294,27.828916,29.060474,1.66129
min,1.000000,180.00000,1.000000,5.000000,5.000000,10.000000,20.000000,5.000000,1.00000
25%,184.750000,330.00000,50.000000,55.000000,50.000000,49.750000,50.000000,45.000000,2.00000
50%,364.500000,450.00000,65.000000,75.000000,70.000000,65.000000,70.000000,65.000000,3.00000
75%,539.250000,515.00000,80.000000,100.000000,90.000000,95.000000,90.000000,90.000000,5.00000
max,721.000000,780.00000,255.000000,190.000000,230.000000,194.000000,230.000000,180.000000,6.00000


Respuesta:
- El data set tiene 800 filas las cuales se componen de 13 columnas (#, Name, Type1, Type2, Total, HP, Attack, Defense, Sp.Atk, Sp.Def, Speed, Generation, Legendary) donde los tipos de datos para cada uno son respectivamente (int64, str, str, str, int64, int64, int64, int64, int64, int64, int64, int64, bool).

- Dentro de este existen valores nulos para la columna "Type2" (386 valores), ya que no todos los pokemones se componen de dos tipos.
##### A continuación se muestra una tabla para describir cada dato:

| Columna  | Descripción                           |
|----------|---------------------------------------|
|#         | ID del pokémon dentro de la pokedex   |
|Name      | Nombre del pokémon                    |
|Type1     | Tipo principal del pokémon            |
|Type2     | Tipo secundario                       |
|Total     | Suma de los atributos de combate      |
|HP        | Puntos de vida del pokémon            |
|Attack    | Puntos de ataque                      |
|Defense   | Puntos de defensa                     |
|Sp.Atk    | Ataque especial                       |
|Sp.Def    | Defensa especial                      |
|Speed     | Velocidad del pokémon                 |
|Generation| Número de generacion al cual pertenece|
|Legendary | True si es legandario, false si no    |

### Preparación del dataset

Realice una preparación del dataset según lo que considere necesario para el entrenamiento de un modelo de clasificación. Justifique las decisiones que tome en este proceso.

In [5]:
# Asignamos features y el objetivo
# NOTAR:
# 1. Se obvia "Total" porque es la suma de los datos -> Redundancia
# 2. Las otras features no consideradas son irrelevantes
features = ['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed']
X = data[features].values.astype('float32')
y = data['Legendary'].values.astype('float32')
# .values.astype('float32') para tener "Numpys arrays" en vez de "Pandas series"

In [ ]:
# Escalamos los datos - Esto se hace ya que los datos se encuentran en escalas distintas
# Entonces, normalizamos para que la red converja mejor
X = StandardScaler().fit_transform(X)

In [7]:
# definimos el dataset, personalisado (extraido desde luisllanca/pytorch-tutorial)
class DataPokemon(torch_data.Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.from_numpy(X.astype('float32'))
        self.y = torch.from_numpy(y.astype('float32')).unsqueeze(1)
        
    def __getitem__(self, idx: int):
        return (self.X[idx], self.y[idx])
    
    def __len__(self):
        return len(self.y)

dataset = DataPokemon(X, y)

In [ ]:
# Split de los datos - Dividimos en Entrenamiento, validation y test. (70 / 15 / 15)
train_set, valid_set, test_set = torch_data.random_split(dataset, [560, 120, 120], generator=torch.Generator().manual_seed(1234))
# Loaders - En train_loader hay shuffle=True para evitar que la red aprenda patrones, en los demás false porque solo se evalúa
train_loader = torch_data.DataLoader(train_set, shuffle=True, batch_size=32)
valid_loader = torch_data.DataLoader(valid_set, shuffle=False, batch_size=64)
test_loader = torch_data.DataLoader(test_set, shuffle=False, batch_size=64)
# (extraido igual desde luisllanca/pytorch-tutorial)

### Definición del modelo  
Defina al menos 3 arquitecturas de redes neuronales simples (MLP) para el problema de clasificación. Justifique las decisiones que tome en la definición de cada arquitectura. Las definiciones se deben hacer en un archivo ```models.py``` e importarlas en este cuadernillo. Debe seleccionar "la mejor" arquitectura para el entrenamiento, y justificar su elección.

### Definición de optimizador y función de costo  
Defina un optimizador y una función de costo adecuado para el entrenamiento del modelo. Justifique sus decisiones.

### Entrenamiento del modelo
Entrene el modelo seleccionado utilizando el dataset preparado. Justifique las decisiones que tome en el proceso de entrenamiento, incluyendo la selección de hiperparámetros, el número de épocas, el tamaño del batch, etc.

### Evaluación del modelo
Evalúe el modelo utilizando métricas adecuadas para este problema de clasificación. Justifique la selección de las métricas utilizadas y discuta los resultados obtenidos. 

### Preguntas finales
1. Sobre la matriz de confusión, interprete los resultados obtenidos. Con sus palabras defina que significa cada tipo de error. ¿Elegiría a Pokémon ubicados en FP o FN para su equipo?
2. Busque un caso mal clasificado por el modelo, e interprete por qué cree que el modelo se equivocó en ese caso.
3. ¿Cúal fue el mayor desafío que enfrentó al realizar esta tarea? ¿Cómo lo solucionó?



### IA Generativa

Con el fin de ocupar IA Generativa de manera responsable, se les pide que respondan a las siguientes preguntas:
1. ¿Utilizó alguna herramienta de IA Generativa para realizar esta tarea? En caso afirmativo, indique cuál o cuáles herramientas utilizó.
2. ¿En qué parte o partes de la tarea utilizó estas herramientas?